# 🛰️ Cartografia de Focos de Queimadas em UCs do Pará (NGEO / IDEFLOR-Bio)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa/blob/main/notebooks/colab_runner.ipynb)

Pipeline automatizado para monitoramento, detecção e cartografia institucional de **focos de calor/incêndio (BDQueimadas/INPE)** e **mosaicos orbitais Sentinel-2 Harmonized (Google Earth Engine)** em Unidades de Conservação do Estado do Pará.

Gera automaticamente 3 pranchas no padrão oficial **A4 Paisagem (297 x 210 mm)**:
1. **Falsa Cor (SWIR-NIR-Red)**: Realce de cicatrizes e áreas queimadas.
2. **Cor Natural (RGB)**: Mosaico de superfície sem nuvens.
3. **NBR (Normalized Burn Ratio)**: Índice espectral de severidade do fogo.

### 1. Instalação de Dependências e Setup do Repositório

In [ ]:
# Instalar pacotes necessários
!pip install -q earthengine-api geopandas rasterio matplotlib pillow

import os
import sys
from pathlib import Path

# Se estiver rodando no Google Colab, clonar ou apontar para o repositório
REPO_NAME = "cartografia-queimadas-ucs-pa"
if not Path(REPO_NAME).exists() and not Path("src").exists():
    print(f"Clonando o repositório {REPO_NAME}...")
    !git clone https://github.com/NGEO-IDEFLOR-Bio/cartografia-queimadas-ucs-pa.git

if Path(REPO_NAME).exists():
    os.chdir(REPO_NAME)

sys.path.append(os.getcwd())
print(f"✓ Diretório de trabalho configurado: {os.getcwd()}")

### 2. Autenticação no Google Earth Engine (GEE)

> ⚠️ **ATENÇÃO:** O projeto padrão configurado abaixo é `ee-samuelsantosambientalcourse`.
> Caso você compartilhe este notebook com colegas de trabalho, oriente cada usuário a **alterar para o seu próprio projeto Google Cloud** associado ao Earth Engine, caso contrário a inicialização da API não funcionará corretamente.

In [ ]:
#@title Autenticação GEE { run: "auto" }
GCP_PROJECT = "ee-samuelsantosambientalcourse" #@param {type:"string"}

import ee

# Autenticação nativa do GEE com o projeto Google Cloud informado
try:
    ee.Initialize(project=GCP_PROJECT)
    print(f"✓ Google Earth Engine inicializado com sucesso no projeto: {GCP_PROJECT}")
except Exception as e:
    print(f"! Inicialização direta pendente ({e}). Solicitando autenticação interativa no Colab...")
    ee.Authenticate()
    ee.Initialize(project=GCP_PROJECT)
    print(f"✓ Google Earth Engine autenticado e inicializado com sucesso no projeto: {GCP_PROJECT}!")

### 3. Painel Interativo de Parâmetros Operacionais

In [ ]:
#@title Parâmetros de Análise Cartográfica { run: "auto" }
NOME_UC = "Área de Proteção Ambiental do Marajó" #@param ["Área de Proteção Ambiental do Marajó", "Área de Proteção Ambiental da Ilha de Combu", "Área de Proteção Ambiental do Lago de Tucuruí", "Área de Proteção Ambiental Triunfo do Xingu", "Área de Proteção Ambiental de Algodoal-Maiandeua", "Área de Proteção Ambiental de São Geraldo do Araguaia – APA Araguaia", "Área de Proteção Ambiental da Região Metropolitana de Belém", "Área de Proteção Ambiental Paytuna", "Parque Estadual Monte Alegre – PEMA", "Parque Estadual Charapucu", "Parque Estadual da Serra dos Martírios/Andorinhas", "Parque Estadual do Utinga “Camillo Vianna”", "Parque Estadual Ambiental das Árvores Gigantes da Amazônia (PAGAM)", "Floresta Estadual do Paru", "Floresta Estadual do Trombetas", "Floresta Estadual do Iriri", "Flota Estadual de Faro", "Estação Ecológica do Grão-Pará (ESEC)", "Estação Ecológica Mamuru (ESEC)", "Reserva Biológica do Maicuru (REBIO)", "Refúgio de Vida Silvestre Metrópole da Amazônia (REVIS)", "Refúgio de Vida Silvestre Padre Sérgio Tonetto", "Refúgio de Vida Silvestre Rios São Benedito e Azul  (REVIS)", "Refúgio de Vida Silvestre Tabuleiro do Embaubal (REVIS)", "Reserva de Desenvolvimento Sustentável Alcobaça", "Reserva de Desenvolvimento Sustentável Campo das Mangabas", "Reserva de Desenvolvimento Sustentável Pucuruí-Ararão", "Reserva de Desenvolvimento Sustentável Vitória de Souzel", "Monumento Natural Atalaia (MONA)"] {allow-input: true}
DATA_INICIO = "2025-08-01" #@param {type:"date"}
DATA_FIM = "2025-12-31" #@param {type:"date"}
FONTE_FOCOS = "BDQUEIMADAS" #@param ["BDQUEIMADAS", "projects/ee-samuelsantosambientalcourse/assets/bdqueimadas_2025-08-01_2025-12-31"] {allow-input: true}
#@markdown *Dica: Deixe 'BDQUEIMADAS' para baixar automaticamente os dados oficiais do INPE para o Pará no período selecionado.*
BUFFER_METROS = 0 #@param {type:"number"}

print("Parâmetros configurados:")
print(f" • Projeto GCP / GEE: {GCP_PROJECT}")
print(f" • Unidade de Conservação: {NOME_UC}")
print(f" • Período: {DATA_INICIO} até {DATA_FIM}")
print(f" • Fonte de Focos: {FONTE_FOCOS}")

### 4. Execução do Pipeline Cartográfico (Geração das 3 Pranchas)

In [ ]:
from src.exporter import processar_e_exportar_todos_os_mapas

resultado = processar_e_exportar_todos_os_mapas(
    nome_uc=NOME_UC,
    data_inicio=DATA_INICIO,
    data_fim=DATA_FIM,
    focos_source=FONTE_FOCOS,
    buffer_uc_metros=BUFFER_METROS
)

### 5. Pré-Visualização das Pranchas Geradas e Download

In [ ]:
from IPython.display import Image, display

pranchas = resultado.get("pranchas", {})

for chave, info in pranchas.items():
    print(f"\n=======================================================")
    print(f" {info['titulo']}")
    print(f" PNG: {info['png']}")
    print(f" PDF: {info['pdf']}")
    print(f"=======================================================")
    display(Image(filename=info["png"], width=900))